# 21 · Does the final system flag some groups of students more than others? (PERSUADE 2.0 demographics)

**Why.** Liang et al. (2023) found that GPT detectors flagged essays by non-native English writers as AI far more often than essays by native writers. Our detector is a fine-tuned model on US grade 6-12 essays, not a perplexity detector, and nothing in the project so far says how it treats English learners or other groups: DAIGT-V2 has no information about who wrote an essay. The public **PERSUADE 2.0** release (Crossley et al., 2024; CC BY 4.0) does: per essay, the writer's English-learner status, grade, gender, race/ethnicity, economic disadvantage and disability status. Our human `persuade_corpus` essays are PERSUADE essays, so an exact text match recovers the labels (`scripts/build_persuade_demographics.py`).

**Everything in the "Plan" below was written and committed before any false-flag rate was computed by group.** Only group *sizes* had been counted (to judge feasibility).

## Plan

**Data and match.** The two public files (training file, 15,594 essays; test file, distributed as a protected archive whose access details are in the PERSUADE README, 10,402 essays) were downloaded and matched to our human essays by exact text after collapsing whitespace: **all 25,996 `persuade_corpus` essays matched** (26,000 rows including four duplicate texts). The 1,375 human essays from the original Kaggle file do not match exactly but share `dup_cluster` groups with matched essays; their labels are propagated through the cluster (the writer's essay is the same essay). After that 27,063 of 27,371 human essays carry demographics. The per-essay mapping stays in git-ignored `data/external/`; only aggregates are committed.

**Population.** Human essays of the **`test` split and the `heldout_prompt` split** (6,755 essays; 6,601 with demographics). Not the training essays (the models were trained on them) and not the val essays (they are the reference of the p scale, so their p-values are circular). The held-out-prompt essays were **scored once**, by the final check (`20_final_heldout_check.ipynb`); **their scores are reused here, nothing is re-scored**: this is a *post-hoc breakdown of data already scored*, not a second held-out test, and it uses no new held-out information for any choice. The test essays are scored with the same final ensemble and calibration.

**The system.** The final 3-seed ensemble (mean of p-values), operating point **A** (flag if p_ens <= 0.00235, cannot tell up to 0.01032) and **B** (flag if p_ens <= 0.00093), exactly as in `19`/`20`. For context, the **old single-model** rule (seed 42, flag at logit >= 4.666) on the same essays, to see whether the move to p-value thresholds changed any disparity.

**Primary comparison (declared now):** **English-learner status "Yes" versus "No"**, the **false-flag rate at operating point A**, pooled over the test and held-out-prompt humans. Reported: both rates with 95% Wilson intervals, the **difference** (Yes minus No, percentage points) and the **rate ratio** with 95% cluster-bootstrap intervals (resampling whole `dup_cluster` groups, 1,000 resamples, seed 42), and the same comparison on each source of essays separately (test; held-out prompts), so that a result is not one split's. Essays with unknown status are reported separately and excluded from the comparison.

**Secondary (all reported, none selected):** for every other attribute (gender, grade level 6-12, race/ethnicity, economic disadvantage, disability status), per group: n, % flagged and % cannot tell at A, % flagged at B, with Wilson intervals, and the difference against the largest group (cluster bootstrap). There are many groups, so **no result will be singled out because it looks extreme**; a group with a high point estimate and a wide interval is reported as such.

**Selection check.** Matched versus unmatched human essays (a handful) and ELL-unknown versus known: do they differ in score? Reported, not corrected.

**What this can and cannot show (declared in advance).** The evaluation set holds only about **340 English-learner essays** against about 6,100 non-learner essays, and the flag rate is below 1%, so **a handful of flagged essays decides the result**. A non-significant difference is **not** evidence of no disparity; the intervals say which disparities can be ruled out (an upper bound on the ratio), and I will report that. "English learner" here is a US school-record classification of grade 6-12 students, not a TOEFL test-taker, so this is **not a replication of Liang et al.** The groups overlap (an English learner may also be economically disadvantaged), and no intersectional analysis is done. These are statements about the **system's behaviour**, not about any individual.

**Data governance.** PERSUADE 2.0 is public (CC BY 4.0, Crossley, Baffour, Tian, Franklin, Benner & Boser, 2024, Assessing Writing 61). The raw files and the per-essay demographic mapping are not committed; the committed results are group-level counts and rates.

## Results

In [1]:
import os
os.environ['HF_HUB_OFFLINE'] = '1'
import sys
sys.path.append('..')

import numpy as np
import pandas as pd
from src.data import load_splits
from src.evaluate import wilson_interval, ClusterSampler
from src.ensemble import PEnsemble

SEED, N_BOOT = 42, 1000
TAU = {'A': (0.00235, 0.01032), 'B': (0.00093, 0.01032)}
pd.set_option('display.width', 230); pd.set_option('display.max_colwidth', 100)
data = load_splits()
COLS = ['ell_status', 'gender', 'grade_level', 'race_ethnicity', 'economically_disadvantaged', 'student_disability_status']
dm = pd.read_csv('../data/external/persuade_demographics.csv', index_col='row_id')
h = data[data['label'] == 0].join(dm[COLS])
known = h[h['gender'].notna() | h['ell_status'].notna()].groupby('dup_cluster')[COLS].first()      # propagate labels inside dup_cluster groups
miss = h['gender'].isna() & h['ell_status'].isna()
h.loc[miss, COLS] = known.reindex(h.loc[miss, 'dup_cluster']).to_numpy()
print('human essays with demographics:', int(h['gender'].notna().sum()), 'of', len(h))

human essays with demographics: 27063 of 27371


In [2]:
# scores of the final system: the test humans with the final ensemble and calibration; the held-out-prompt humans from the final check (reused, not re-scored)
vt = data[data['split'].isin(['val', 'test'])]
s42 = pd.read_csv('../results/15_scores.csv', index_col='row_id')['roberta']
f43 = pd.read_csv('../results/17_final/orig_s43_scores.csv', index_col='row_id')['logit']; f44 = pd.read_csv('../results/17_final/orig_s44_scores.csv', index_col='row_id')['logit']
vh = vt[(vt['split'] == 'val') & (vt['label'] == 0)].index
ens = PEnsemble(np.stack([s42.loc[vh], f43.loc[vh], f44.loc[vh]], axis=1), 'mean')
th = h[h['split'] == 'test']
X = np.stack([s42.loc[th.index], f43.loc[th.index], f44.loc[th.index]], axis=1)
test_part = pd.DataFrame({'p_ens': ens.p_ens(X), 'logit_s42': X[:, 0]}, index=th.index)
ho20 = pd.read_csv('../results/20_heldout_scores.csv', index_col='row_id')
hp = h[h['split'] == 'heldout_prompt']
ho_part = ho20.loc[hp.index, ['p_ens', 'logit_s42']]
ev = pd.concat([h.loc[test_part.index].join(test_part), h.loc[ho_part.index].join(ho_part)])
for op, (f, b) in TAU.items():
    ev[f'flag_{op}'] = ev['p_ens'] <= f; ev[f'band_{op}'] = (ev['p_ens'] > f) & (ev['p_ens'] <= b)
ev['old_1pct'] = ev['logit_s42'] >= 4.666; ev['old_05pct'] = ev['logit_s42'] >= 7.796
for c in COLS[:1] + COLS[1:]:
    ev[c] = ev[c].where(ev[c].notna(), 'unknown').astype(str).str.strip().replace({'': 'unknown', 'nan': 'unknown', 'None': 'unknown'})
ev['grade_level'] = ev['grade_level'].str.replace('.0', '', regex=False)
print(len(ev), 'humans in the evaluation set;', ev.groupby('split').size().to_dict())
print('overall flagged at A: %.2f%%, at B: %.2f%%, in the band at A: %.2f%%' % (100 * ev['flag_A'].mean(), 100 * ev['flag_B'].mean(), 100 * ev['band_A'].mean()))

6755 humans in the evaluation set; {'heldout_prompt': 4464, 'test': 2291}
overall flagged at A: 0.49%, at B: 0.37%, in the band at A: 1.95%


### Primary comparison: English-learner status, false-flag rate at operating point A

In [3]:
def wil(k, n):
    lo, hi = wilson_interval(int(k), int(n)); return f'{100 * k / n:.2f} [{100 * lo:.2f}, {100 * hi:.2f}] ({int(k)}/{int(n)})'
def compare(df, col, g1, g0, flag='flag_A', n_boot=N_BOOT):
    a, b = (df[col] == g1).to_numpy(), (df[col] == g0).to_numpy(); f = df[flag].to_numpy()
    smp = ClusterSampler(df['dup_cluster'].to_numpy()); rng = np.random.default_rng(SEED)
    def stat(rows):
        r1 = f[rows][a[rows]].mean() if a[rows].any() else np.nan; r0 = f[rows][b[rows]].mean() if b[rows].any() else np.nan
        return r1, r0
    r1, r0 = stat(np.arange(len(df)))
    diffs, ratios = [], []
    for _ in range(n_boot):
        x1, x0 = stat(smp.draw(rng)); diffs.append(100 * (x1 - x0)); ratios.append(x1 / x0 if x0 > 0 else np.nan)
    ratios = np.array(ratios); dd = np.array(diffs)
    return {'group 1': f'{g1} ({int(a.sum())} essays, {int(f[a].sum())} flagged)', 'rate 1 %': wil(f[a].sum(), a.sum()), 'group 0': f'{g0} ({int(b.sum())} essays, {int(f[b].sum())} flagged)',
            'rate 0 %': wil(f[b].sum(), b.sum()), 'difference (pts)': f'{100 * (r1 - r0):+.2f} [{np.nanpercentile(dd, 2.5):+.2f}, {np.nanpercentile(dd, 97.5):+.2f}]',
            'rate ratio': f'{r1 / r0:.2f} [{np.nanpercentile(ratios, 2.5):.2f}, {np.nanpercentile(ratios, 97.5):.2f}]' if r0 > 0 else 'n/a (no flags in group 0)'}
rows = []
known_ell = ev[ev['ell_status'].isin(['Yes', 'No'])]
for name, d in [('POOLED (test + held-out prompts)', known_ell), ('test split', known_ell[known_ell['split'] == 'test']), ('held-out prompts', known_ell[known_ell['split'] == 'heldout_prompt'])]:
    for op in ('A', 'B'):
        rows.append({'essays': name, 'operating point': op, **compare(d, 'ell_status', 'Yes', 'No', flag=f'flag_{op}')})
ell = pd.DataFrame(rows).set_index(['essays', 'operating point'])
ell

group 1                   rate 1 %                       group 0                     rate 0 %      difference (pts)         rate ratio
essays                           operating point                                                                                                                                                            
POOLED (test + held-out prompts) A                Yes (340 essays, 0 flagged)  0.00 [0.00, 1.12] (0/340)  No (6120 essays, 33 flagged)  0.54 [0.38, 0.76] (33/6120)  -0.54 [-0.73, -0.36]  0.00 [0.00, 0.00]
                                 B                Yes (340 essays, 0 flagged)  0.00 [0.00, 1.12] (0/340)  No (6120 essays, 25 flagged)  0.41 [0.28, 0.60] (25/6120)  -0.41 [-0.56, -0.25]  0.00 [0.00, 0.00]
test split                       A                Yes (212 essays, 0 flagged)  0.00 [0.00, 1.78] (0/212)   No (1927 essays, 2 flagged)   0.10 [0.03, 0.38] (2/1927)  -0.10 [-0.26, +0.00]  0.00 [0.00, 0.00]
                                 B                Yes (212 essays, 0 flagged)  0.00 [0.00, 1.78] (0/212)   No (1927 essays, 2 flagged)   0.10 [0.03, 0.38] (2/1927)  -0.10 [-0.26, +0.00]  0.00 [0.00, 0.00]
held-out prompts                 A                Yes (128 essays, 0 flagged)  0.00 [0.00, 2.91] (0/128)  No (4193 essays, 31 flagged)  0.74 [0.52, 1.05] (31/4193)  -0.74 [-1.00, -0.48]  0.00 [0.00, 0.00]
                                 B                Yes (128 essays, 0 flagged)  0.00 [0.00, 2.91] (0/128)  No (4193 essays, 23 flagged)  0.55 [0.37, 0.82] (23/4193)  -0.55 [-0.76, -0.33]  0.00 [0.00, 0.00]

In [4]:
# supplementary (added after the plan, labelled): the same comparison adjusted for prompt (stratified difference), and the old single-model rule
def prompt_adjusted(df, flag='flag_A', n_boot=N_BOOT):
    code_ = pd.factorize(df['prompt_name'])[0]; L = (df['ell_status'] == 'Yes').to_numpy().astype(int); F = df[flag].to_numpy().astype(int); P = code_.max() + 1
    smp = ClusterSampler(df['dup_cluster'].to_numpy()); rng = np.random.default_rng(SEED)
    def stat(rows):
        cell = code_[rows] * 2 + L[rows]; n = np.bincount(cell, minlength=2 * P).reshape(P, 2); k = np.bincount(cell, weights=F[rows], minlength=2 * P).reshape(P, 2)
        ok = (n > 0).all(axis=1); w = n[ok].sum(axis=1)
        return 100 * float((w * (k[ok, 1] / n[ok, 1] - k[ok, 0] / n[ok, 0])).sum() / w.sum())
    pt = stat(np.arange(len(df))); dr = [stat(smp.draw(rng)) for _ in range(n_boot)]
    return f'{pt:+.2f} [{np.percentile(dr, 2.5):+.2f}, {np.percentile(dr, 97.5):+.2f}]'
sup = [{'rule': 'final ensemble, operating point A', 'ELL yes: flagged %': wil(known_ell.loc[known_ell['ell_status'] == 'Yes', 'flag_A'].sum(), (known_ell['ell_status'] == 'Yes').sum()),
        'ELL no: flagged %': wil(known_ell.loc[known_ell['ell_status'] == 'No', 'flag_A'].sum(), (known_ell['ell_status'] == 'No').sum()), 'prompt-adjusted difference (pts)': prompt_adjusted(known_ell, 'flag_A')},
       {'rule': 'old single model (seed 42), logit >= 4.666', 'ELL yes: flagged %': wil(known_ell.loc[known_ell['ell_status'] == 'Yes', 'old_1pct'].sum(), (known_ell['ell_status'] == 'Yes').sum()),
        'ELL no: flagged %': wil(known_ell.loc[known_ell['ell_status'] == 'No', 'old_1pct'].sum(), (known_ell['ell_status'] == 'No').sum()), 'prompt-adjusted difference (pts)': prompt_adjusted(known_ell, 'old_1pct')}]
pd.DataFrame(sup).set_index('rule')

,ELL yes: flagged %,ELL no: flagged %,prompt-adjusted difference (pts)
rule,,,
"final ensemble, operating point A","0.00 [0.00, 1.12] (0/340)","0.54 [0.38, 0.76] (33/6120)","-0.53 [-0.72, -0.36]"
"old single model (seed 42), logit >= 4.666","0.88 [0.30, 2.56] (3/340)","3.06 [2.65, 3.52] (187/6120)","-1.93 [-3.24, -0.34]"


### Secondary: every other attribute, every group (none selected)

In [5]:
def group_table(attr, order=None):
    d = ev[ev[attr] != 'unknown'] if attr != 'ell_status' else ev
    groups = order or sorted(d[attr].unique())
    ref = d[attr].value_counts().idxmax()
    rows = []
    for g in groups:
        s = d[d[attr] == g]
        if len(s) == 0: continue
        cmp_ = compare(d[d[attr].isin([g, ref])], attr, g, ref) if g != ref else None
        rows.append({'attribute': attr, 'group': g, 'n': len(s), '% flagged, A': wil(s['flag_A'].sum(), len(s)), '% cannot tell, A': wil(s['band_A'].sum(), len(s)), '% flagged, B': wil(s['flag_B'].sum(), len(s)),
                     '% flagged, old single model': wil(s['old_1pct'].sum(), len(s)), f'difference vs largest group ({ref}), pts at A': cmp_['difference (pts)'] if cmp_ else '(reference group)'})
    return pd.DataFrame(rows)
sec = pd.concat([group_table('gender'), group_table('grade_level', ['6', '8', '9', '10', '11', '12']), group_table('race_ethnicity'), group_table('economically_disadvantaged'),
                 group_table('student_disability_status'), group_table('ell_status', ['Yes', 'No', 'unknown'])], ignore_index=True)
sec.set_index(['attribute', 'group'])

n                 % flagged, A              % cannot tell, A                 % flagged, B   % flagged, old single model  \
attribute                  group                                                                                                                                                             
gender                     F                                    3407  0.65 [0.43, 0.98] (22/3407)   2.20 [1.76, 2.75] (75/3407)  0.47 [0.29, 0.76] (16/3407)  3.29 [2.74, 3.94] (112/3407)   
                           M                                    3194  0.34 [0.19, 0.62] (11/3194)   1.78 [1.38, 2.31] (57/3194)   0.28 [0.15, 0.53] (9/3194)   2.47 [1.99, 3.07] (79/3194)   
grade_level                6                                     134    0.00 [0.00, 2.79] (0/134)     0.00 [0.00, 2.79] (0/134)    0.00 [0.00, 2.79] (0/134)     0.00 [0.00, 2.79] (0/134)   
                           8                                     976    0.10 [0.02, 0.58] (1/976)     0.72 [0.35, 1.47] (7/976)    0.10 [0.02, 0.58] (1/976)     0.92 [0.49, 1.74] (9/976)   
                           9                                    2582   0.12 [0.04, 0.34] (3/2582)   0.58 [0.35, 0.96] (15/2582)   0.08 [0.02, 0.28] (2/2582)   1.36 [0.98, 1.88] (35/2582)   
                           10                                   1059   0.57 [0.26, 1.23] (6/1059)   1.61 [1.00, 2.56] (17/1059)   0.47 [0.20, 1.10] (5/1059)   2.36 [1.60, 3.46] (25/1059)   
                           11                                   1675  1.37 [0.92, 2.05] (23/1675)   5.43 [4.45, 6.62] (91/1675)  1.01 [0.63, 1.62] (17/1675)  7.16 [6.02, 8.50] (120/1675)   
                           12                                     47     0.00 [0.00, 7.56] (0/47)     2.13 [0.38, 11.11] (1/47)     0.00 [0.00, 7.56] (0/47)     2.13 [0.38, 11.11] (1/47)   
race_ethnicity             American Indian/Alaskan Native         36     0.00 [0.00, 9.64] (0/36)      0.00 [0.00, 9.64] (0/36)     0.00 [0.00, 9.64] (0/36)      0.00 [0.00, 9.64] (0/36)   
                           Asian/Pacific Islander                471    1.91 [1.01, 3.59] (9/471)    6.16 [4.32, 8.70] (29/471)    1.49 [0.72, 3.04] (7/471)   8.92 [6.66, 11.83] (42/471)   
                           Black/African American               1219   0.33 [0.13, 0.84] (4/1219)   1.39 [0.87, 2.22] (17/1219)   0.16 [0.05, 0.60] (2/1219)   2.05 [1.39, 3.01] (25/1219)   
                           Hispanic/Latino                      1614   0.25 [0.10, 0.64] (4/1614)   1.49 [1.00, 2.20] (24/1614)   0.19 [0.06, 0.55] (3/1614)   1.80 [1.25, 2.57] (29/1614)   
                           Two or more races/Other               236    0.00 [0.00, 1.60] (0/236)     1.69 [0.66, 4.28] (4/236)    0.00 [0.00, 1.60] (0/236)     2.12 [0.91, 4.86] (5/236)   
                           White                                3025  0.53 [0.33, 0.86] (16/3025)   1.92 [1.49, 2.47] (58/3025)  0.43 [0.25, 0.73] (13/3025)   2.98 [2.43, 3.64] (90/3025)   
economically_disadvantaged Economically disadvantaged           1455   0.34 [0.15, 0.80] (5/1455)   1.58 [1.06, 2.36] (23/1455)   0.27 [0.11, 0.70] (4/1455)   2.34 [1.68, 3.25] (34/1455)   
                           Not economically disadvantaged       2205  1.09 [0.73, 1.61] (24/2205)   4.17 [3.41, 5.09] (92/2205)  0.82 [0.52, 1.29] (18/2205)  5.35 [4.49, 6.37] (118/2205)   
student_disability_status  Identified as having disability       383    0.26 [0.05, 1.46] (1/383)     2.09 [1.06, 4.07] (8/383)    0.26 [0.05, 1.46] (1/383)    2.61 [1.42, 4.74] (10/383)   
                           Not identified as having disability  3283  0.85 [0.59, 1.23] (28/3283)  3.26 [2.70, 3.92] (107/3283)  0.64 [0.42, 0.98] (21/3283)  4.33 [3.68, 5.08] (142/3283)   
ell_status                 Yes                                   340    0.00 [0.00, 1.12] (0/340)     1.18 [0.46, 2.99] (4/340)    0.00 [0.00, 1.12] (0/340)     0.88 [0.30, 2.56] (3/340)   
                           No                                   6120  0.54 [0

### Selection checks

In [6]:
unk = ev['gender'] == 'unknown'
chk = pd.DataFrame({'essays': [int((~unk).sum()), int(unk.sum()), int((ev['ell_status'].isin(['Yes', 'No'])).sum()), int((ev['ell_status'] == 'unknown').sum())],
                    'flagged at A %': [wil(ev.loc[~unk, 'flag_A'].sum(), (~unk).sum()), wil(ev.loc[unk, 'flag_A'].sum(), unk.sum()),
                                       wil(ev.loc[ev['ell_status'].isin(['Yes', 'No']), 'flag_A'].sum(), ev['ell_status'].isin(['Yes', 'No']).sum()), wil(ev.loc[ev['ell_status'] == 'unknown', 'flag_A'].sum(), (ev['ell_status'] == 'unknown').sum())],
                    'median p_ens': [float(ev.loc[~unk, 'p_ens'].median()), float(ev.loc[unk, 'p_ens'].median()), float(ev.loc[ev['ell_status'].isin(['Yes', 'No']), 'p_ens'].median()), float(ev.loc[ev['ell_status'] == 'unknown', 'p_ens'].median())]},
                   index=['matched (has demographics)', 'no match', 'ELL status known', 'ELL status unknown'])
chk.round(4)

,essays,flagged at A %,median p_ens
matched (has demographics),6601,"0.50 [0.36, 0.70] (33/6601)",0.3351
no match,154,"0.00 [0.00, 2.43] (0/154)",0.3325
ELL status known,6460,"0.51 [0.36, 0.72] (33/6460)",0.3338
ELL status unknown,295,"0.00 [0.00, 1.29] (0/295)",0.3774


In [7]:
pd.concat({'primary_ELL': ell.reset_index(), 'secondary': sec, 'selection_checks': chk.reset_index()}, names=['table']).to_csv('../results/21_fairness_by_group.csv', index=False)
print('saved results/21_fairness_by_group.csv (aggregates only)')

saved results/21_fairness_by_group.csv (aggregates only)


## Interpretation

*(Written after the results were committed, `9da1980`. The plan above was committed before any group rate was computed, `8ec9543`.)*

### Primary result: English learners

**No English-learner essay was flagged: 0 of 340 (0.00%, 95% Wilson interval [0.00, 1.12]) at operating point A, against 33 of 6,120 (0.54% [0.38, 0.76]) for non-learners**; the same 0 of 340 at operating point B, and 0 of 212 on the test split and 0 of 128 on the held-out prompts separately. Under equal rates one would expect 1.8 flagged English-learner essays, so seeing none is not surprising (probability 0.16; Fisher exact p = 0.42): **there is no evidence of a difference in either direction.** What the data **can** exclude: an English-learner false-flag rate much above the interval's upper end, **1.12%, about 2.1 times the non-learner rate**; an excess smaller than that cannot be ruled out. *The bootstrap interval for the difference and the ratio printed in the table (for example "-0.54 [-0.73, -0.36]" and a ratio interval of [0, 0]) is too narrow to read: with zero events in a group it does not vary, so it reflects only the non-learner rate's uncertainty. The Wilson interval above is the honest one.*

- **The old single-model rule gives the same direction:** English learners 0.88% [0.30, 2.56] (3 of 340) against 3.06% [2.65, 3.52] (187 of 6,120); prompt-adjusted difference -1.93 [-3.24, -0.34] points. The move to p-value thresholds did not change the direction.
- **The cannot-tell band:** English learners 1.18% [0.46, 2.99] (4 of 340), non-learners 2.08% [1.75, 2.46].
- **This is not a replication of Liang et al. (2023)**, and does not contradict it: their detectors were perplexity-based, their non-native writers were TOEFL test-takers, and our English learners are US school-record classifications of grade 6-12 students writing on prompts the model trained on. It does suggest that **this fine-tuned model's false flags are not driven by "non-native-looking" writing**, which fits the earlier finding that the humans it flags are the more polished and formulaic ones (notebooks 14, 16b); it does not show that.

### Secondary results (all groups reported in the table; none selected)

- **Grade:** grade 11 looks high (1.37% [0.92, 2.05]) against grade 9 (0.12% [0.04, 0.34]), but this is **prompt mix**: 1,527 of the 1,675 grade-11 essays are "Summer projects", the hardest prompt for the system (1.60% in the held-out check). Adjusted for prompt, grade 11 against grades 8-10 is **-0.73 [-2.76, +0.89]** points: no grade effect. Within Summer projects alone the rate is 1.51% for grade 11 (1,527 essays) and 2.39% for grade 10 (209 essays).
- **Economic disadvantage:** students not classified as economically disadvantaged are flagged more: 1.09% [0.73, 1.61] (24 of 2,205) against 0.34% [0.15, 0.80] (5 of 1,455); raw +0.74 points, prompt-adjusted **+0.56 [+0.06, +1.06]**; the interval just excludes zero (one of many comparisons).
- **Race/ethnicity:** Asian/Pacific Islander students have the highest point estimate, 1.91% [1.01, 3.59] (9 of 471), against White 0.53% [0.33, 0.86], Black 0.33% [0.13, 0.84] and Hispanic 0.25% [0.10, 0.64]; prompt-adjusted Asian/Pacific Islander minus White **+1.06 [-0.18, +2.65]**: not distinguishable from zero. Not singled out.
- **Gender:** girls 0.65% [0.43, 0.98] against boys 0.34% [0.19, 0.62]; prompt-adjusted **+0.33 [-0.00, +0.67]**.
- **Disability status:** students with an identified disability 0.26% [0.05, 1.46] (1 of 383) against 0.85% [0.59, 1.23].
- **A pattern, not a finding:** the groups with higher flag rates (not economically disadvantaged, girls, no identified disability, Asian/Pacific Islander) tend to be the groups whose writing scores are higher on average in PERSUADE; the groups with lower rates (English learners, economically disadvantaged, disability) the opposite. This is consistent with false flags falling on polished, formulaic writing; it is not tested here (writing quality was not controlled) and several differences have intervals that include zero.

### Selection checks

The 154 human essays with no match are not distinguishable (0 flagged, median p 0.333 against 0.335 for matched essays), and the 295 essays with unknown English-learner status look like the known ones (0 flagged; median p 0.377 against 0.334).

### Limits

- **Few events:** only 33 flagged human essays have a known English-learner status; the English-learner group is 340 essays. Differences of up to about a factor of two cannot be excluded for any group with a few hundred essays.
- **Many comparisons** (about 25 groups), several confounded with prompt: grade, and possibly race and economic status, track which prompt a student answered. Only the three adjusted comparisons above were checked, and they were added after seeing the table.
- **No intersections** (an English learner can also be economically disadvantaged) and no control for writing quality.
- **Post-hoc use of the held-out prompts** (scores reused from the final check, nothing re-scored; no choice depends on them), and an evaluation set that excludes the essays the models were trained on.
- These are statements about the **system's behaviour on groups**, not about any individual essay or student.

**For the documents** (proposed, not applied): the model card's "Not tested: essays by non-native writers" can become "Partly tested: US school English learners, 0 of 340 flagged at the final threshold (upper bound 1.12%); no evidence of a higher rate, power limited; not a test of TOEFL-style non-native writers". It should **not** claim the system is fair across groups: the groups with the highest rates (not economically disadvantaged, Asian/Pacific Islander) are not the ones usually feared, but the burden of false flags is uneven and the cause is unknown.